# Extractor de Características con CNN
Vamos a entrenar una CNN sencilla para comprimir las imágenes de MNIST en un embedding de 16 dimensiones.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


In [6]:
class MNISTExtractor(nn.Module):
    def __init__(self, embedding_dim=16):
        super(MNISTExtractor, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)  # Embedding denso de 16 dim
        )
        self.classifier = nn.Linear(embedding_dim, 10)

    def forward(self, x):
        features = self.features(x)
        emb = self.embedding(features)
        out = self.classifier(emb)
        return out

    def get_embedding(self, x):
        features = self.features(x)
        return self.embedding(features)

model = MNISTExtractor(embedding_dim=16).to(device)

In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 1
print("Empezando entrenamiento...")
for epoch in range(epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        if batch_idx % 200 == 0:
            print(f"Epoch: {epoch+1} [{batch_idx*len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

Empezando entrenamiento...
Epoch: 1 [0/60000] Loss: 2.306520
Epoch: 1 [12800/60000] Loss: 0.175026
Epoch: 1 [25600/60000] Loss: 0.438622
Epoch: 1 [38400/60000] Loss: 0.189949
Epoch: 1 [51200/60000] Loss: 0.170849


In [9]:
def extract_embs(dataset):
    loader = torch.utils.data.DataLoader(dataset, batch_size=1000, shuffle=False)
    all_embs = []
    all_targets = []
    model.eval()
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            emb = model.get_embedding(data)
            all_embs.append(emb.cpu())
            all_targets.append(target)
    return torch.cat(all_embs), torch.cat(all_targets)

print("Extrayendo embeddings...")
train_embs, train_labels = extract_embs(train_dataset)
test_embs, test_labels = extract_embs(test_dataset)

torch.save(train_embs, "train_embeddings.pt")
torch.save(train_labels, "train_labels.pt")
torch.save(test_embs, "test_embeddings.pt")
torch.save(test_labels, "test_labels.pt")
print("Completado! Ficheros de embeddings guardados como .pt")

Extrayendo embeddings...
Completado! Ficheros de embeddings guardados como .pt
